# Post-Processing — Classical Pipeline

**Input:** `anomaly_scores.csv` from the anomaly detection step.  
**Output:** `classical_anomaly_report.csv` — the ranked transit anomaly report.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import config as cfg
import warnings
warnings.filterwarnings("ignore")

## 1. Load Anomaly Scores

We load the full dataset with all baseline features and anomaly scores computed in the previous notebook.

In [2]:
ANOMALY_DIR = cfg.IO_DIR / "anomaly_detection"
df = pd.read_csv(ANOMALY_DIR / "anomaly_scores.csv")
df["date"] = pd.to_datetime(df["date"])

print(f"Shape: {df.shape}")
print(f"Consensus anomalies from models: {df['consensus_anomaly'].sum()} ({df['consensus_anomaly'].mean():.1%})")

Shape: (3449, 80)
Consensus anomalies from models: 75 (2.2%)


## 2. Business Rules

Statistical models flag anomalies based on mathematical patterns, but not all statistical anomalies are operationally meaningful. The post-processing layer applies domain-specific rules to highlight observations that match known risk patterns:

| Rule | Logic | Rationale |
|------|-------|-----------|
| **Flag rate spike** | `flag_rate_dev_ratio7 > 3` | Flag rate exceeded 3× its recent rolling average |
| **Entries spike** | `entries_dev_ratio7 > 3` | Passenger volume exceeded 3× its recent rolling average |
| **High seasonal residual** | `abs(entries_residual_z) > 2` | Entries deviate by more than 2σ from the monthly norm for this route |
| **High alarm density** | `alarm_density_per_entry > 95th percentile` | Disproportionately many alarm cases relative to traffic volume |

The thresholds are up to our linking, we have looked up for a standard setup

In [3]:
# Define business rules
df["rule_flag_spike"] = (df["flag_rate_dev_ratio7"] > 3).astype(int)
df["rule_entries_spike"] = (df["entries_dev_ratio7"] > 3).astype(int)
df["rule_high_residual_z"] = (df["entries_residual_z"].abs() > 2).astype(int)

alarm_95 = df["alarm_density_per_entry"].quantile(0.95)
df["rule_high_alarm"] = (df["alarm_density_per_entry"] > alarm_95).astype(int)

df["rule_any"] = df[["rule_flag_spike", "rule_entries_spike",
                      "rule_high_residual_z", "rule_high_alarm"]].max(axis=1)

print("Business rules triggered:")
print(f"  Flag rate > 3× baseline:     {df['rule_flag_spike'].sum()}")
print(f"  Entries > 3× baseline:       {df['rule_entries_spike'].sum()}")
print(f"  Seasonal residual |z| > 2:   {df['rule_high_residual_z'].sum()}")
print(f"  Alarm density > 95th pct:    {df['rule_high_alarm'].sum()}")
print(f"  Any rule triggered:          {df['rule_any'].sum()}")

Business rules triggered:
  Flag rate > 3× baseline:     179
  Entries > 3× baseline:       23
  Seasonal residual |z| > 2:   149
  Alarm density > 95th pct:    125
  Any rule triggered:          430


## 3. Final Anomaly Label

The final anomaly label combines model consensus and business rules:

```
final_anomaly = consensus_anomaly OR rule_any
```

This ensures that:
- Statistically unusual observations are captured even if no single business rule fires
- Operationally meaningful patterns are captured even if the statistical models don't flag them

In [4]:
df["final_anomaly"] = ((df["consensus_anomaly"] == 1) | (df["rule_any"] == 1)).astype(int)

print(f"Total anomalies: {df['final_anomaly'].sum()} / {len(df)} ({df['final_anomaly'].mean():.1%})")
print(f"  From consensus model only: {((df['consensus_anomaly'] == 1) & (df['rule_any'] == 0)).sum()}")
print(f"  From business rules only:  {((df['consensus_anomaly'] == 0) & (df['rule_any'] == 1)).sum()}")
print(f"  Flagged by both:           {((df['consensus_anomaly'] == 1) & (df['rule_any'] == 1)).sum()}")

Total anomalies: 487 / 3449 (14.1%)
  From consensus model only: 57
  From business rules only:  412
  Flagged by both:           18


## File Export

In [5]:
ANOMALY_DIR = cfg.IO_DIR / "anomaly_detection"
ANOMALY_DIR.mkdir(parents=True, exist_ok=True)
 
output_path = ANOMALY_DIR / "classical_post_processed.csv"
df.to_csv(output_path, index=False)